# 小红书帖子 · 第一阶段探索分析（结构化步骤）

**推荐**：在项目根目录 `/Users/yilin/Desktop/project/robot_failure` 启动 Jupyter，使 `phase1` 包可被导入。

```bash
cd /Users/yilin/Desktop/project/robot_failure
source .venv/bin/activate
jupyter notebook notebooks/phase1_structured.ipynb
```

**流水线对应代码**（便于脚本复现）：

| 步骤 | 模块 | 说明 |
|------|------|------|
| A | `phase1/preprocess.py` | 读表、合并 `post_category`、一级/二级去重、统一长表 |
| B | `phase1/features.py` | 角色/拟人/边界/玩梗词典特征 |
| C | `phase1/analysis.py` | 互动图、聚类、导出辅助 CSV/图 |
| D | `phase1/reports.py` | `data_quality`、摘要报告、codebook 草案 |
| 一键 | `phase1/pipeline.py` 或 `python run_phase1.py` | 全流程 |

## 0. 环境与项目根目录

若 Kernel 工作目录不是项目根，可取消注释下面 `os.chdir(...)` 一行并改成你的路径。

In [ ]:
from pathlib import Path
import os

# PROJECT_ROOT = Path("/Users/yilin/Desktop/project/robot_failure")
# os.chdir(PROJECT_ROOT)

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / "phase1" / "pipeline.py").exists(), "请在 robot_failure 根目录运行 Kernel"
assert (PROJECT_ROOT / "小红书帖子数据.xlsx").exists(), "缺少 Excel 数据文件"

print("ROOT =", PROJECT_ROOT)

## 1. 数据预处理（步骤 A）

- 一级评论：`一级评论id` 去重  
- 二级评论：`二级评论id` 去重，父键 `parent_comment_id` = 一级 id  
- 互动字段强制数值化，避免后续 `mean` 报错

In [ ]:
from phase1.preprocess import (
    build_level1,
    build_level2,
    coerce_engagement,
    load_raw_frames,
    merge_post_category,
    unified_comments,
)

main_df, counts = load_raw_frames()
merged = merge_post_category(main_df, counts)

l1 = coerce_engagement(build_level1(merged))
l2 = coerce_engagement(build_level2(merged))
unified = coerce_engagement(unified_comments(l1, l2))

len(main_df), merged['帖子id'].nunique(), len(l1), len(l2), len(unified)

## 2. 落盘清洗表（可选）

与 `python run_phase1.py` 输出路径一致：`output/phase1/`

In [ ]:
from phase1.config import OUT

OUT.mkdir(parents=True, exist_ok=True)
l1.to_csv(OUT / "clean_l1_comments.csv", index=False)
l2.to_csv(OUT / "clean_l2_comments.csv", index=False)
unified.to_csv(OUT / "clean_comments_unified.csv", index=False)
print("已写入", OUT)

## 3. 词典与特征（步骤 B）

词表定义见 `phase1/lexicons.py`；可在本格之后自行改词典再重跑。

In [ ]:
from phase1.features import apply_lexicons

enriched = apply_lexicons(unified)
enriched.head(2)

## 4. 分析导出（步骤 C）

含互动汇总、角色热力图、拟人共现、玩梗与边界、TF-IDF+KMeans 主题（轻量）。**首次画图**会配置中文字体（`phase1/config.py`）。

In [ ]:
from phase1.config import configure_matplotlib
from phase1.analysis import (
    ensure_dirs,
    boundary_outputs,
    interaction_map,
    meme_outputs,
    personhood_outputs,
    role_aggregate,
    topic_clusters,
)

configure_matplotlib()
ensure_dirs()

interaction_map(l1)
role_aggregate(enriched)
personhood_outputs(enriched)
meme_outputs(enriched)
boundary_outputs(enriched)
topic_clusters(
    unified["content"].tolist(),
    unified["post_category"].tolist(),
    top_n=min(8000, len(unified)),
)

enriched.to_csv(OUT / "comments_enriched.csv", index=False)
print("figures & csv -> output/phase1")

## 5. 报告（步骤 D）

In [ ]:
from phase1.reports import (
    build_phase1_summary,
    write_codebook_suggestions,
    write_data_quality,
)

write_data_quality(merged, l1, l2, unified)
build_phase1_summary(enriched, l1)
write_codebook_suggestions(enriched, l1)

## 6. 一键等价调用（可选）

与终端 `python run_phase1.py` 等价。

In [ ]:
# from phase1.pipeline import run_phase1_pipeline
# result = run_phase1_pipeline()
# result["enriched"].head()